# Checking fold results of all methods in summary.json files generated by nnUNet

In [ ]:
# imports
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from scipy.stats import wilcoxon
import SimpleITK as sitk

In [ ]:
# path to predicted data
data_path = '../nnUNet_data/nnUNet_raw/Dataset361_Menisci/iwoai_internal_results/'

# list subfolders at this path
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir()]
print(subfolders)

In [ ]:
# what metrics are there?
json_path = os.path.join(data_path, 'clahe_predsTs', 'summary.json')

with open(json_path) as f:
    data = json.load(f)

# extract all metrics
data["metric_per_case"][1]

In [ ]:
# functions to take json file and return array of dice scores, or return hausdorff distance
def get_dice_scores(json_file):
    with open(json_file) as f:
        data = json.load(f)

    # Extract Dice scores from "metric_per_case"
    dice_scores = [
        case["metrics"]["1"]["Dice"] for case in data["metric_per_case"]
    ]

    return dice_scores


def get_hausdorff_distances(json_file):
    with open(json_file) as f:
        data = json.load(f)

    # Extract Hausdorff distances from "metric_per_case"
    hausdorff_distances = [
        case["metrics"]["1"]["Hausdorff_95"] for case in data["metric_per_case"]
    ]

    return hausdorff_distances

In [ ]:
# get only the subfolders that contain '_fold_'
subfolders = [f for f in subfolders if '_fold_' in f]

subfolders

In [ ]:
# reorder the subfolders using the order of the methods (starts with)
method_order = ["zscore", "rescale", "clip_and_rescale", "hist_eq", "clahe", "nyul", "gmm"]

# Sort by checking which method each subfolder starts with
subfolders = sorted(
    subfolders,
    key=lambda x: next((method_order.index(m) for m in method_order if x.lower().startswith(m.lower())), float('inf'))
)
subfolders

In [ ]:
# for each folder, go through each fold subfolder (fold0 - fold5) and get average dice scores

# make dictionary to hold the 5 fold averages for each method
fold_averages = {method: [None] * 5 for method in method_order}

for folder in subfolders:
    method_name = folder.split('_fold_')[0]  # Extract method name from folder name
    
    # go through all 5 folds
    for fold in range(5):
        json_path = os.path.join(data_path, folder, f'fold{fold}', 'summary.json')
        if not os.path.exists(json_path):
            print(f"Warning: {json_path} does not exist.")
            continue
        
        dice_scores = get_dice_scores(json_path)
        fold_averages[method_name][fold] = np.mean(dice_scores)

In [ ]:
fold_averages_df = pd.DataFrame(fold_averages)
fold_averages_df.index.name = 'Fold'

In [ ]:
fold_averages_df.describe()

In [ ]:
means = fold_averages_df.mean()
stds = fold_averages_df.std()

plt.figure(figsize=(10, 5))
plt.bar(means.index, means.values, yerr=stds.values, capsize=5)
plt.ylabel("Dice Score")
plt.title("5-Fold Cross-Validation Performance")
plt.xticks(rotation=45)
# make y start at 0.8
plt.ylim(0.88, 0.90)
plt.tight_layout()
plt.show()

In [ ]:
df_long = fold_averages_df.reset_index().melt(id_vars='Fold', var_name='Method', value_name='Dice Score')

plt.figure(figsize=(10, 6))
sns.stripplot(data=df_long, x='Method', y='Dice Score', jitter=True)
plt.title("Dice Scores Across 5 Folds")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sns.set(style="whitegrid")
sns.pointplot(data=df_long, x='Method', y='Dice Score', errorbar='sd', linestyle='none')
plt.xticks(rotation=45)
# title for fold average distribution across methods
plt.title("Fold Average Distribution Across Methods")
plt.xlabel(None)
plt.tight_layout()
# Save the plot as a PDF
plt.savefig('fold_averages_plot.pdf', bbox_inches='tight', dpi=500, format='pdf')
plt.show()

### Mixed Effects model on internal testing

In [ ]:
# Again read in dice scores for each method and fold
# this time storing in long format (method, fold, image_id, dice_score)

data = []

for folder in subfolders:
    method_name = folder.split('_fold_')[0]  # Extract method name from folder name
    
    # go through all 5 folds
    for fold in range(5):
        json_path = os.path.join(data_path, folder, f'fold{fold}', 'summary.json')
        if not os.path.exists(json_path):
            print(f"Warning: {json_path} does not exist.")
            continue
        
        dice_scores = get_dice_scores(json_path)

        for i, score in enumerate(dice_scores):
            data.append({
                'Method': method_name,
                'Fold': fold,
                'Image_ID': i,
                'Dice_Score': score
            })

# Convert to DataFrame
internal_df_long = pd.DataFrame(data)

In [ ]:
internal_df_long.head()

In [ ]:
# multiply dice scores by 100 to get percentage
internal_df_long['Dice_Score'] = internal_df_long['Dice_Score'] * 100

In [ ]:
# fit mixed effects model to the data
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
# Fit a mixed effects model with a random effects term for fold and Image_ID

internal_df_long["Method"] = pd.Categorical(
    internal_df_long["Method"],
    categories=["zscore", "rescale", "clip_and_rescale", "hist_eq", "clahe", "nyul", "gmm"],
    ordered=True
)

model = mixedlm(
    "Dice_Score ~ Method", 
    internal_df_long, 
    groups=internal_df_long["Image_ID"], 
    re_formula="~1",
    vc_formula={"Fold": "0 + C(Fold)"}
)

result = model.fit()
print(result.summary())

## External Validation on whole of SKMTEA

In [ ]:
data_path = '../nnUNet_data/nnUNet_raw/Dataset361_Menisci/skmtea_external_results/'

# list subfolders at this path
subfolders = [f.name for f in os.scandir(data_path) if f.is_dir()]

# get only the subfolders that contain '_fold_'
subfolders = [f for f in subfolders if '_fold_' in f]

subfolders

In [ ]:
# reorder the subfolders using the order of the methods (starts with)
method_order = ["zscore", "rescale", "clip_and_rescale", "hist_eq", "clahe", "nyul", "gmm"]
# Sort by checking which method each subfolder starts with
subfolders = sorted(
    subfolders,
    key=lambda x: next((method_order.index(m) for m in method_order if x.lower().startswith(m.lower())), float('inf'))
)
subfolders

In [ ]:
# for each folder, go through each fold subfolder (fold0 - fold5) and get average dice scores

# make dictionary to hold the 5 fold averages for each method
fold_averages = {method: [None] * 5 for method in method_order}

for folder in subfolders:
    method_name = folder.split('_fold_')[0]  # Extract method name from folder name
    
    # go through all 5 folds
    for fold in range(5):
        json_path = os.path.join(data_path, folder, f'fold{fold}', 'summary.json')
        if not os.path.exists(json_path):
            print(f"Warning: {json_path} does not exist.")
            continue
        
        dice_scores = get_dice_scores(json_path)
        fold_averages[method_name][fold] = np.mean(dice_scores)

In [ ]:
external_fold_averages_df = pd.DataFrame(fold_averages)
external_fold_averages_df.index.name = 'Fold'
external_fold_averages_df.describe()

In [ ]:
df_long = external_fold_averages_df.reset_index().melt(id_vars='Fold', var_name='Method', value_name='Dice Score')
plt.figure(figsize=(10, 6))
sns.stripplot(data=df_long, x='Method', y='Dice Score', jitter=True)
plt.title("Dice Scores Across 5 Folds (External)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sns.set(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.pointplot(data=df_long, x='Method', y='Dice Score', errorbar='sd', linestyle='none')
plt.xticks(rotation=45)
plt.title("Fold Average Distribution Across Methods (External)")
plt.xlabel(None)
plt.tight_layout()
# Save the plot as a PDF
plt.savefig('external_fold_averages_plot.pdf', bbox_inches='tight', dpi=500, format='pdf')
plt.show()

# Attempt at Mixed Effects Model for external results

In [ ]:
# Again read in dice scores for each method and fold
# this time storing in long format (method, fold, image_id, dice_score)

data = []

for folder in subfolders:
    method_name = folder.split('_fold_')[0]  # Extract method name from folder name
    
    # go through all 5 folds
    for fold in range(5):
        json_path = os.path.join(data_path, folder, f'fold{fold}', 'summary.json')
        if not os.path.exists(json_path):
            print(f"Warning: {json_path} does not exist.")
            continue
        
        dice_scores = get_dice_scores(json_path)

        for i, score in enumerate(dice_scores):
            data.append({
                'Method': method_name,
                'Fold': fold,
                'Image_ID': i,
                'Dice_Score': score
            })

# Convert to DataFrame
external_df_long = pd.DataFrame(data)

In [ ]:
external_df_long.head()

In [ ]:
# turn dice scores into percentage
external_df_long['Dice_Score'] = external_df_long['Dice_Score'] * 100

In [ ]:
# fit mixed effects model to the data
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm
# Fit a mixed effects model with a random effects term for fold and Image_ID

external_df_long["Method"] = pd.Categorical(
    external_df_long["Method"],
    categories=["zscore", "rescale", "clip_and_rescale", "hist_eq", "clahe", "nyul", "gmm"],
    ordered=True
)

model = mixedlm(
    "Dice_Score ~ Method", 
    external_df_long, 
    groups=external_df_long["Image_ID"],
    re_formula="~1",
    vc_formula={"Fold": "0 + C(Fold)"}
)

result = model.fit()
print(result.summary())

# save result.summary table as a pdf